# Webscrapping Player Positions

This notebook generates the csv used in the main Jupyter notebook (Basketball Positions). The data is webscrapped from https://www.sports-reference.com/cbb/. 

# Import Libraries & Modules

In [ ]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

# Setup Global Variables

In [ ]:
earliest_season = 2005
latest_season = 2025

# Webscrape

## School IDs

In [ ]:
url = 'https://www.sports-reference.com/cbb/schools/'
headers = {'User-Agent': 'Mozilla/5.0'}
max_retries = 5
initial_delay = 15  # seconds
for attempt in range(max_retries):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 429:
            print(f"429 Too Many Requests. Retrying in {initial_delay} seconds...")
            time.sleep(initial_delay)
            initial_delay *= 2  # Exponential backoff
            continue # Retry with longer delay
        response.raise_for_status()  # Raise other HTTP errors
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table', id='NCAAM_schools')
        if table:
            break # Success: move onto extracting data
        else:
            print("Table not found. The structure might have changed.")
            break
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        time.sleep(initial_delay)
else:
    print("Max retries reached. Exiting...")
headers = [th.get_text(strip=True) for th in table.find_all('thead')[0].find_all('th')]
rows = []
for row in table.find('tbody').find_all('tr'):
    if row.get('class') == ['thead']:
        continue
    data = [cell.get_text(strip=True) for cell in row.find_all(['th', 'td'])]
    if data:
        rows.append(data)
schools_df = pd.DataFrame(rows, columns=headers)

In [ ]:
# Map the difference between school index and html url
exceptions_map = {
    "Alabama A&M": "alabama-am",
    "Albany (NY)": "albany-ny",
    "Allegheny Gators": "allegheny",
    "Amherst Lord Jeffs": "amherst",
    "Armstrong Pirates": "armstrong",
    "Augusta Jaguars": "augusta-state",
    "Augustana (IL) Vikings": "augusta-il",
    "Baldwin-Wallace Yellow Jackets": "baldwin-wallace",
    "Baltimore Super Bees": "baltimore",
    "Beloit Buccaneers": "beloit",
    "Birmingham-Southern Panthers": "birmingham-southern",
    "Bloomsburg Huskies": "bloomsburg",
    "Brooklyn Bulldogs": "brooklyn",
    "Cal State Los Angeles Golden Eagles": "cal-state-los-angeles",
    "Canterbury College": "central-normal-college",  
    "Carleton College Knights": "carleton-college",
    "Carnegie Mellon Tartans": "carnegie-",
    "Case Western Reserve Spartans": "case-western-reserve",
    "Catholic Cardinals": "catholic",
    "Centenary (LA) Gents": "centenary-la",
    "Central Missouri Mules": "central-missouri",
    "Central Pennsylvania College Knights": "central-pennsylvania-college",
    "Centre (KY) Colonels": "centre-ky",
    "Chicago Maroons": "chicago",
    "City College of New York Beavers": "city-college-of-new-york",
    "College of New Jersey Lions": "college-of-new-jersey",
    "Colorado College Tigers": "colorado-college",
    "Colorado School of Mines Orediggers": "colorado-school-of-mines",
    "Concordia Seminary Preachers": "concordia-seminary",
    "Cotner College": "cotner-university",
    "Dakota Wesleyan Tigers": "dakota-wesleyan",
    "Denison Big Red": "denison",
    "DePauw Tigers": "depauw",
    "Dickinson College Red Devils": "dickinson-college",
    "East Central Tigers": "east-central",
    "East Texas A&M": "texas-am-commerce",
    "Emporia State Hornets": "emporia-state",
    "Ensign College": "latter-day-saints-university",
    "FDU": "fairleigh-dickinson",
    "Florida A&M": "florida-am",
    "Franklin Grizzlies": "franklin",
    "Geneva Golden Tornadoes": "geneva",
    "Gettysburg Bullets": "gettysburg",
    "Grinnell Pioneers": "grinnell",
    "Grove City Wolverines": "grove-city",
    "Hamline Pipers": "hamline",
    "Hardin-Simmons Cowboys": "hardin-simmons",
    "Hartford Hawks": "hartford",
    "Haskell (KS) Fighting Indians": "haskell-indian-nations-institute",
    "Hiram Terriers": "hiram",
    "Hope Flying Dutchmen": "hope",
    "Houston Christian": "houston-baptist",
    "Illinois Wesleyan Titans": "illinois-wesleyan",
    "IU Indy": "iupui",
    "John Carroll Blue Streaks": "john-carroll",
    "Kalamazoo Hornets": "kalamazoo",
    "Kansas City": "missouri-kansas-city",
    "Kentucky Wesleyan Panthers": "kentucky-wesleyan",
    "Lake Forest Foresters": "lake-forest",
    "Lewis Flyers": "lewis",
    "Little Rock": "arkansas-little-rock",
    "Louisiana": "louisiana-lafayette",
    "Loyola (IL)": "loyola-il",
    "Loyola (LA) Wolfpack": "loyola-la",
    "Loyola (MD)": "loyola-md",
    "Macalester Scots": "macalester",
    "Manchester Spartans": "manchester",
    "Marietta Pioneers": "marietta",
    "Massachusetts Institute of Technology Engineers": "massachusetts-institute-of-technology",  
    "Merchant Marine Mariners": "merchant-marine-academy",
    "Miami (FL)": "miami-fl",
    "Miami (OH)": "miami-oh",
    "Millikin Big Blue": "millikin",
    "Millsaps Majors": "millsaps",
    "Minnesota A&M Aggies": "minnesota-am",
    "Morris Brown Wolverines": "morris-brown",
    "Mount St. Mary's": "mount-st-marys",
    "Mount Union Purple Raiders": "mount-union",
    "Muhlenberg Mules": "muhlenberg",
    "Muskingum Fighting Muskies": "muskingum",
    "NC State": "north-carolina-state",
    "Nebraska Wesleyan Prairie Wolves": "nebraska-wesleyan",
    "New York University Violets": "new-york-university",
    "Newberry Wolves": "newberry",
    "North Carolina A&T": "north-carolina-at",
    "North Central (IL) Cardinals": "north-central",
    "Northeastern Illinois Golden Eagles": "northeastern-illinois",
    "Northwest Missouri State Bearcats": "northwest-missouri-state",
    "Oberlin Yeomen": "oberlin",
    "Ohio Wesleyan Battling Bishops": "ohio-wesleyan",
    "Oklahoma City Chiefs": "oaklahoma-city",
    "Omaha": "nebraska-omaha",
    "Phillips Haymakers": "phillips",  
    "Pittsburg State Gorillas": "pittsburg-state",
    "Pratt Institute Cannoneers": "pratt-institute",
    "Purdue Fort Wayne": "ipfw",
    "Queens (NC)": "queens-nc",
    "Regis (CO) Rangers": "regis-co",
    "Rensselaer Engineers": "rensselaer",  
    "Ripon Red Hawks": "ripon",
    "Roanoke Maroons": "roanoke",
    "Rochester (NY) Yellowjackets": "rochester-ny",
    "Rose-Hulman Fightin' Engineers": "rose-hulman",  
    "Saint Francis (PA)": "saint-francis-pa",
    "Saint Joseph's": "saint-josephs",
    "Saint Mary's (CA)": "saint-marys-ca",
    "Saint Peter's": "saint-peters",
    "Sam Houston": "sam-houston-state",
    "Savannah State Tigers": "savannah-state",
    "Scranton Royals": "scranton",
    "Sewanee Tigers": "sewanee",
    "Southwestern (KS) Moundbuilders": "southwestern-ks",
    "Southwestern (TX) Pirates": "southwestern-tx",
    "Springfield Pride": "springfield",
    "St. Bonaventure": "st-bonaventure",
    "St. Francis (NY) Terriers": "st-francis-ny",
    "St. John's (NY)": "st-johns-ny",
    "St. John's College (OH)": "st-johns-college-oh",
    "St. Lawrence Saints": "st-lawrence",
    "St. Thomas": "st-thomas-mn",
    "Stephen F. Austin": "stephen-f-austin",
    "Stevens Institute Ducks": "stevens-institute",
    "SUNY-Potsdam Bears": "suny-potsdam",
    "Swarthmore Garnet": "swarthmore",
    "TCU": "texas-christian",
    "Texas A&M": "texas-am",
    "Texas A&M-Corpus Christi": "texas-am-corpus-christi",
    "Texas Wesleyan Rams": "texas-wesleyan",
    "Texas-Rio Grande Valley": "texas-pan-american",
    "The Citadel": "citadel",
    "Trinity (CT) Bantams": "trinity-ct",
    "Trinity (TX) Tigers": "trinity-tx",
    "U.S. International Gulls": "alliant-international",
    "UAB": "alabama-birmingham",
    "UC Davis": "california-davis",
    "UC Irvine": "california-irvine",
    "UC Riverside": "california-riverside",
    "UC San Diego": "california-san-diego",
    "UC Santa Barbara": "california-santa-barbara",
    "UCF": "central-florida",
    "UNC Asheville": "north-carolina-asheville",
    "UNC Greensboro": "north-carolina-greensboro",
    "UNC Wilmington": "north-carolina-wilmington",
    "Union (NY) Dutchmen": "union-ny",
    "UT Arlington": "texas-arlington",
    "Utah Tech": "dixie-state",
    "UTEP": "texas-el-paso",
    "Utica Pioneers": "utica",
    "UTSA": "texas-san-antonio",
    "Wabash Little Giants": "wabash",
    "Washburn Ichabods": "washburn",
    "Washington & Jefferson Presidents": "washington-jefferson",
    "Washington & Lee Generals": "washington-lee",
    "Washington (MO) Bears": "washington-mo",
    "Washington College Shoremen": "washington-college",
    "Wayne State (MI) Warriors": "wayne-state-mi",
    "Wesleyan (CT) Cardinals": "wesleyan-ct",
    "West Chester Golden Rams": "west-chester",
    "West Texas A&M Buffaloes": "west-texas-am",
    "Western Colorado Mountaineers": "western-state",
    "Westminster (MO) Blue Jays": "westminster-mo",
    "Westminster (PA) Titans": "westminster-pa",
    "Wheaton (IL) Thunder": "wheaton-il",
    "Whittier Poets": "whittier",
    "Widener Pride": "widener",
    "William & Mary": "william-mary",
    "Williams Ephs": "williams",
    "Wisconsin-Stevens Point Pointers": "wisconsin-stevens-point",
    "Wisconsin-Superior Yellowjackets": "wisconsin-superior",
    "Wittenberg Tigers": "wittenberg",
    "Wooster Fighting Scots": "wooster",
    "WPI Engineers": "worcester-tech"
}

In [ ]:
schools_df['School'] = schools_df['School'].replace(exceptions_map)
schools_df['To'] = schools_df['To'].astype('int32')
schools_df = schools_df[schools_df['To'] > (earliest_season - 1)].reset_index() # Teams must exist in the intended time period
schools_df = schools_df['School']
schools_df = schools_df.str.replace(' ', '-').str.lower()  # Must be lowercase to work in html url
schools_df

## Player Stats

In [ ]:
player_stats = pd.DataFrame()
max_retries = 10
initial_delay = 5  # seconds
for year in range(earliest_season, latest_season+1):
    print(year) # Show the progress of code
    for school in schools_df:
        url = f"https://www.sports-reference.com/cbb/schools/{school}/men/{year}.html"
        retries = 0
        delay = initial_delay
        while retries < max_retries:
            try:
                headers = {'User-Agent': 'Mozilla/5.0'}
                response = requests.get(url, headers=headers, timeout=10)
                if response.status_code == 429:
                    time.sleep(delay)
                    delay *= 2  # Exponential backoff
                    retries += 1
                    continue # Retry with longer delay
                if response.status_code != 200:
                    break  # Move to the next school; failed to retrieve data
                soup = BeautifulSoup(response.content, 'html.parser')
                table = soup.find('table', id='players_per_game')
                if table is None:
                    print(f"No data found for {school} in {year}.")
                    break  # Move to the next school
                headers = [th.get_text(strip=True) for th in table.find('thead').find_all('th')]
                rows = []
                for row in table.find('tbody').find_all('tr'):
                    if row.get('class') == ['thead']:
                        continue
                    data = [cell.get_text(strip=True) for cell in row.find_all(['th', 'td'])]
                    if data:
                        rows.append(data)
                df = pd.DataFrame(rows, columns=headers)
                drop = ['Rk', 'Awards', 'G']
                df = df.drop(columns=[col for col in drop if col in df.columns], errors='ignore')
                df['Season'] = int(year)
                df['School'] = school
                player_stats = pd.concat([player_stats, df], ignore_index=True)  # Rolling merge of data across schools and years
                break  # Move to the next school
            except requests.RequestException as e:
                print(f"Error for {school}: {e}")
                time.sleep(initial_delay)
                break
player_stats

In [ ]:
player_stats.to_csv("Player_Stats.csv", index=False)  # Adjust CSV file path